# sweep-hparam-distribution — worked example 3: int_uniform for integer hyperparameters

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sweep-hparam-distribution`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a hyperparameter is an integer count — such as number of layers or hidden size — use `'distribution': 'int_uniform'` with integer `min` and `max` values. This instructs wandb to sample only integer values within the range, rather than continuous floats that would need rounding.

## Worked solution

**Step 1 — Why not `uniform`?**
`'distribution': 'uniform'` samples continuous floats. Passing a float like 3.7 as `num_layers` would likely cause a type error in your model constructor, or silently truncate, producing unexpected behavior.

**Step 2 — Use `int_uniform`.**
`{'distribution': 'int_uniform', 'min': 2, 'max': 10}` tells wandb to sample integers in [2, 10] inclusive on both ends.

**Step 3 — Keep min and max as Python ints.**
Passing floats like `min=2.0` can cause subtle schema issues in some wandb versions. Use bare int literals.

**Step 4 — When to use `values` instead.**
If the valid integer choices are sparse or non-contiguous (e.g., `[1, 3, 7]`) or follow a specific pattern (e.g., powers of 2), `'values': [1, 3, 7]` is more appropriate than `int_uniform`.

In [ ]:
def num_layers_spec(min_layers: int = 2, max_layers: int = 10) -> dict:
    """Integer count: int_uniform samples integers in [min, max] inclusive."""
    return {
        'distribution': 'int_uniform',
        'min': int(min_layers),   # ensure int, not float
        'max': int(max_layers),
    }

def hidden_size_spec() -> dict:
    """Another integer hparam — hidden dimension of MLP layers."""
    return {
        'distribution': 'int_uniform',
        'min': 64,
        'max': 512,
    }

# Contrast: when to use 'values' instead
def hidden_size_powers_of_2() -> dict:
    """Powers-of-2 hidden sizes: discrete list is cleaner than int_uniform."""
    return {'values': [64, 128, 256, 512]}

# Demonstrate
print('num_layers spec:', num_layers_spec())
print('hidden_size spec:', hidden_size_spec())
print('hidden_size (powers-of-2):', hidden_size_powers_of_2())